# Lesson 03 - Các mẫu thiết kế Agentic

Trong bài học này, chúng ta khám phá ba mẫu thiết kế nền tảng để xây dựng các tác nhân AI hiệu quả:

1. **Hướng dẫn rõ ràng cho tác nhân** — Tạo các câu nhắc định nghĩa vai trò chính xác để hướng dẫn hành vi của tác nhân  
2. **Đầu ra có cấu trúc với các mô hình Pydantic** — Đảm bảo các tác nhân trả về dữ liệu có thể dự đoán và được xác thực  
3. **Tác nhân đảm nhận một trách nhiệm duy nhất** — Thiết kế các tác nhân tập trung chỉ làm tốt một việc

Chúng ta sẽ áp dụng từng mẫu cho kịch bản **trình đề xuất điểm đến du lịch**, xây dựng dần dần một hệ thống có thể đề xuất địa điểm, kiểm tra khả năng sẵn có, và xử lý logistics.


## Cài đặt


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Mẫu 1: Hướng dẫn rõ ràng cho Đại lý

Mẫu có tác động lớn nhất cũng là đơn giản nhất: viết hướng dẫn rõ ràng, chi tiết cho đại lý của bạn.

Hướng dẫn tốt xác định:
- **Ai** là đại lý (nhân vật và giọng điệu)
- **Cái gì** đại lý nên làm (trách nhiệm từng bước)
- **Cách** đại lý nên hành xử (hạn chế và phong cách)

Dưới đây, chúng tôi tạo một đại lý trợ lý du lịch với hướng dẫn rõ ràng điều chỉnh mọi phản hồi mà nó tạo ra.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Mẫu 2: Đầu ra có cấu trúc với các mô hình Pydantic

Văn bản tự do hữu ích cho cuộc trò chuyện, nhưng các hệ thống phía dưới cần dữ liệu có cấu trúc.  
Bằng cách kết hợp **mô hình Pydantic** với một **hàm công cụ**, chúng ta có thể:

- Định nghĩa một lược đồ chính xác cho đầu ra của tác nhân
- Tự động xác thực các phản hồi
- Tích hợp kết quả của tác nhân vào logic ứng dụng một cách đáng tin cậy

Chúng tôi cũng giới thiệu một công cụ trả về chi tiết điểm đến để tác nhân dựa vào dữ liệu thực để đưa ra khuyến nghị.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Mẫu 3: Các đại lý có trách nhiệm đơn lẻ

Các công việc phức tạp được lợi từ việc chia nhỏ công việc cho nhiều đại lý tập trung, mỗi người chịu một trách nhiệm duy nhất:

- Một **Chuyên gia Điểm đến** biết về địa điểm và sự sẵn có
- Một **Người lên kế hoạch Logistics** xử lý chuyến bay, khách sạn và hành trình

Điều này phản ánh nguyên tắc kỹ thuật phần mềm *tách biệt các mối quan tâm* — mỗi đại lý dễ dàng hơn để kiểm tra, bảo trì và cải tiến độc lập.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Tóm tắt

Trong bài học này, chúng tôi đã áp dụng ba mẫu thiết kế tác nhân vào một kịch bản đề xuất du lịch:

| Mẫu | Ý tưởng chính | Lợi ích |
|---|---|---|
| **Hướng dẫn rõ ràng** | Định nghĩa nhân vật, trách nhiệm và giới hạn ngay từ đầu | Hành vi tác nhân nhất quán, đúng thương hiệu |
| **Đầu ra có cấu trúc** | Sử dụng mô hình Pydantic làm định dạng phản hồi | Kết quả được xác thực, có thể đọc bởi máy |
| **Trách nhiệm đơn lẻ** | Giao cho mỗi tác nhân một công việc tập trung | Dễ dàng kiểm thử, bảo trì và kết hợp |

Các mẫu này kết hợp một cách tự nhiên — bạn có thể kết hợp hướng dẫn rõ ràng với đầu ra có cấu trúc bên trong một tác nhân có trách nhiệm đơn lẻ để xây dựng hệ thống mạnh mẽ, sẵn sàng cho môi trường sản xuất.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố từ chối trách nhiệm**:  
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi nỗ lực đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được xem là nguồn chính xác và đáng tin cậy nhất. Đối với các thông tin quan trọng, chúng tôi khuyến nghị sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm đối với bất kỳ sự hiểu nhầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
